Programa el agente que le toque de acuerdo a la asignación. Para cada agente, se debe identificar y explicar: Los estados, el entorno, la política, recompensa y la función de acción. Para el proceso de exploración y explotación se debe aplicar una de las siguientes estrategias: "Selección de acciones con intervalo de confianza" ó "Algoritmo del gradiente".
-----------------
   Crea un agente que juegue rompecabezas deslizante 3x3.

In [21]:
import numpy as np

class Board():
    def __init__(self):
        # El estado objetivo/ganador se usa como base inicial
        self.goal_state = np.array([
            [1, 2, 3],
            [4, 5, 6],
            [7, 8, 0]
        ])
        self.state = np.copy(self.goal_state)

    def _find_zero(self):
        # para encontrar la posición (fila, col) del espacio vacío.
        pos = np.argwhere(self.state == 0)
        return pos[0][0], pos[0][1]

    def valid_moves(self):
    
        #Retorna una lista de tuplas (row, col) con las piezas que se pueden deslizar hacia el espacio vacío.
        z_row, z_col = self._find_zero()
        moves = []
        
        # Direcciones posibles: arriba, abajo, izquierda, derecha
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        
        for dr, dc in directions:
            r, c = z_row + dr, z_col + dc
            if 0 <= r < 3 and 0 <= c < 3:
                moves.append((r, c))     
        return moves

    def update(self, row, col):
    
       # Intercambia la pieza en (row, col) con el espacio vacío. (No se necesita pasar un 'symbol' porque movemos piezas existentes).
        if (row, col) in self.valid_moves():
            z_row, z_col = self._find_zero()
            # Intercambio de posiciones usando la sintaxis de numpy
            self.state[z_row, z_col], self.state[row, col] = self.state[row, col], self.state[z_row, z_col]
        else:
            raise ValueError("¡Movimiento ilegal! La pieza no está junto al espacio vacío.")

    def is_game_over(self):
      
        #Comprueba si el tablero llegó al estado objetivo.Retorna 1 si está resuelto, None si hay que seguir jugando.
        
        if np.array_equal(self.state, self.goal_state):
            return 1 # Rompecabezas resuelto
        
        return None # Seguir jugando

    def reset(self):
        #Reinicia el tablero al estado inicial resuelto.
        self.state = np.copy(self.goal_state)
        
    def shuffle(self, steps=20):
        #Mezcla el tablero usando solo movimientos válidos para asegurar solución.
        for _ in range(steps):
            moves = self.valid_moves()
            # Elige un movimiento válido al azar
            random_move = moves[np.random.choice(len(moves))]
            self.update(*random_move)

In [22]:
from tqdm import tqdm


class Game:
    def __init__(self, agent):
        self.agent = agent
        self.board = Board()

    def selfplay(self, rounds=100, max_steps=100, shuffle_steps=30):
        resoluciones = 0

        for i in tqdm(range(1, rounds + 1)):
            self.board.reset()
            # Mezclamos el tablero para crear el acertijo
            self.board.shuffle(steps=shuffle_steps)
            self.agent.reset()

            game_over = False
            steps = 0

            while not game_over and steps < max_steps:
                # El agente elige la pieza a mover
                action = self.agent.move(self.board)

                # Desplazamos la pieza en el tablero (action es una tupla: row, col)
                self.board.update(action[0], action[1])

                # Guardamos el estado actual en el historial del agente
                self.agent.update(self.board)

                steps += 1

                # Comprobamos si ya se resolvió
                if self.board.is_game_over() is not None:
                    game_over = True
                    resoluciones += 1

            # Al final de la ronda asignamos la recompensa
            self.reward(game_over)

        return f"Rompecabezas resueltos: {resoluciones}/{rounds}"

    def reward(self, solved):
        # Si game_over fue True, significa que se resolvió antes del límite de pasos
        if solved:
            self.agent.reward(1.0)
        else:
            self.agent.reward(0.0)

In [ ]:
class Agent:
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {}  # tabla estado -> valor
        self.alpha = alpha  # learning rate
        self.positions = []  # historial de estados en la partida
        self.prob_exp = prob_exp  # probabilidad de exploracion

        # Inicializamos el estado ganador con un valor alto conocido (1)
        goal_state = str(np.array([[1, 2, 3], [4, 5, 6], [7, 8, 0]]).reshape(9))
        self.value_function[goal_state] = 1.0

    def reset(self):
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()

        # exploracion
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            ix = np.random.choice(len(valid_moves))
            return valid_moves[ix]

        # explotacion
        max_value = -1000
        best_row, best_col = valid_moves[0]

        # Encontrar la posicion actual del cero en el tablero
        z_row, z_col = board._find_zero()

        for row, col in valid_moves:
            # Clonamos el estado para simular el movimiento de deslizamiento
            next_board = board.state.copy()

            # Intercambio de la pieza elegida con el espacio vacio
            next_board[z_row, z_col], next_board[row, col] = (
                next_board[row, col],
                next_board[z_row, z_col],
            )

            next_state = str(next_board.reshape(9))

            # Si el estado no existe en la tabla, por defecto vale 0
            value = self.value_function.get(next_state, 0.0)

            if value >= max_value:
                max_value = value
                best_row, best_col = row, col

        return best_row, best_col

    def update(self, board):
        self.positions.append(str(board.state.reshape(9)))

    def reward(self, reward):
          # al final de la partida (cuando recibimos la recompensa)
        # iteramos por tods los estados actualizando su valor en la tabla
            if self.value_function.get(p) is None:
                self.value_function[p] = 0.0
            self.value_function[p] += self.alpha * (
                reward - self.value_function[p]
            )
            reward = self.value_function[p]

In [25]:
# 1. Creamos un único agente con su probabilidad de exploración inicial
agent = Agent(prob_exp=0.5)
game = Game(agent)

# 3. Entrenamos al agente durante 300 partidas
resultado = game.selfplay(rounds=300)

# 4. Imprimimos el resultado para ver cuántas logró resolver
print(resultado)

100%|██████████| 300/300 [00:19<00:00, 15.27it/s]

Rompecabezas resueltos: 123/300
